In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

import joblib
import json
import os


In [2]:


DATA_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data"
PIPELINE_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\models\\processing"
DOCS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\docs"
FEATURE_METADATA_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\processed\\feature_names.json"

# Create required directories
os.makedirs(PIPELINE_PATH, exist_ok=True)
os.makedirs(DOCS_PATH, exist_ok=True)

RANDOM_STATE = 42


In [4]:
print("Loading training data...")

X_train = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data X_train.csv")
print("✓ X_train loaded:", X_train.shape)


Loading training data...
✓ X_train loaded: (424200, 23)


In [5]:
try:
    with open(FEATURE_METADATA_PATH, "r") as f:
        feature_metadata = json.load(f)
    print("✓ Feature metadata loaded")
except FileNotFoundError:
    feature_metadata = None
    print("⚠ feature_names.json not found — using column analysis instead")


✓ Feature metadata loaded


In [6]:
all_columns = X_train.columns.tolist()

# Encoded categorical columns
encoded_categorical = [col for col in all_columns if col.endswith("_encoded")]

# Numeric columns
numeric_features = [col for col in all_columns if col not in encoded_categorical]

print(f"Numeric features: {len(numeric_features)}")
print(f"Encoded categorical features: {len(encoded_categorical)}")


Numeric features: 17
Encoded categorical features: 6


In [7]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", MinMaxScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, encoded_categorical)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("✓ Preprocessing pipeline created")


✓ Preprocessing pipeline created


In [8]:
preprocessor.fit(X_train)
print("✓ Pipeline fitted on training data")

X_train_transformed = preprocessor.transform(X_train)
print("Transformed shape:", X_train_transformed.shape)


✓ Pipeline fitted on training data
Transformed shape: (424200, 23)


In [9]:
print("NaN values:", np.isnan(X_train_transformed).sum())
print("Infinite values:", np.isinf(X_train_transformed).sum())


NaN values: 0
Infinite values: 0


In [10]:
pipeline_file = f"{PIPELINE_PATH}preprocessing_pipeline.pkl"
joblib.dump(preprocessor, pipeline_file)

print("✓ Pipeline saved at:", pipeline_file)


✓ Pipeline saved at: C:\Users\sneha\Desktop\ecopackai\Data\ml\models\processingpreprocessing_pipeline.pkl


In [11]:
column_groups = {
    "numeric_features": numeric_features,
    "encoded_categorical_features": encoded_categorical,
    "all_features": all_columns,
    "output_feature_count": X_train_transformed.shape[1]
}

column_file = f"C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\docs\\preprocessing_columns.json"
with open(column_file, "w") as f:
    json.dump(column_groups, f, indent=2)

print("✓ Column metadata saved")


✓ Column metadata saved


In [12]:
X_val = pd.read_csv(f"C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data X_val.csv")
X_val_transformed = preprocessor.transform(X_val)

print("Validation transformed shape:", X_val_transformed.shape)


Validation transformed shape: (60600, 23)


In [13]:
sample_df = pd.DataFrame(
    X_train_transformed[:10],
    columns=[f"feature_{i}" for i in range(X_train_transformed.shape[1])]
)

sample_path = f"{DATA_PATH}sample_transformed.csv"
sample_df.to_csv(sample_path, index=False)

print("✓ Sample transformed data saved")


✓ Sample transformed data saved


In [14]:
print("PREPROCESSING PIPELINE COMPLETE ✅")

print("Input features:", len(all_columns))
print("Output features:", X_train_transformed.shape[1])
print("Numeric:", len(numeric_features))
print("Categorical:", len(encoded_categorical))

print("\nNext steps:")
print("1. Train baseline ML models")
print("2. Apply cross-validation")
print("3. Evaluate MAE, RMSE, R²")


PREPROCESSING PIPELINE COMPLETE ✅
Input features: 23
Output features: 23
Numeric: 17
Categorical: 6

Next steps:
1. Train baseline ML models
2. Apply cross-validation
3. Evaluate MAE, RMSE, R²
